In [ ]:
import os
import json
from PIL import Image, ImageDraw, ImageOps

# Function to check if a directory exists
def check_directory(path):
    if not os.path.exists(path):
        raise FileNotFoundError(f"Directory does not exist: {path}")

# Function to process JSON files and extract segmentation for a given category_id
def process_json_files(json_source_path, category_id_to_crop):
    crop_info = {}

    # Check if the source path exists
    check_directory(json_source_path)

    # Iterate through all JSON files in the source directory
    for filename in os.listdir(json_source_path):
        if filename.endswith('.json'):
            json_file_path = os.path.join(json_source_path, filename)
            with open(json_file_path, 'r') as f:
                data = json.load(f)

                # Iterate through annotations and find matching category_id
                for annotation in data['annotations']:
                    if annotation['category_id'] == category_id_to_crop:
                        crop_info[filename] = annotation['segmentation']  # Use segmentation data
                        break  # Only need one match per file

    return crop_info

# Function to create a mask and crop the image based on segmentation
def crop_images(crop_info, image_source_path, image_dest_path):
    # Ensure destination path exists
    os.makedirs(image_dest_path, exist_ok=True)

    for json_filename, segmentations in crop_info.items():
        image_filename = json_filename.replace('.json', '.jpeg')  # Assuming .jpeg images
        image_path = os.path.join(image_source_path, image_filename)

        if os.path.exists(image_path):
            with Image.open(image_path) as img:
                img = ImageOps.exif_transpose(img)  # Correct orientation if needed

                # Create a transparent background image of same size as original
                object_img = Image.new('RGBA', img.size)

                # Create a mask from the segmentation points (same size as image)
                mask = Image.new('L', img.size, 0)  # Create a blank grayscale mask (L mode)
                mask_draw = ImageDraw.Draw(mask)

                # Draw all polygons (segmentations) on the mask
                for segmentation in segmentations:
                    polygon = [(segmentation[i], segmentation[i+1]) for i in range(0, len(segmentation), 2)]
                    mask_draw.polygon(polygon, outline=255, fill=255)  # Fill polygon with white (255)

                # Apply mask to keep only object and make background transparent
                object_img.paste(img.convert('RGBA'), (0, 0), mask=mask)

                # Crop to bounding box of non-zero pixels in mask (object area)
                bbox = mask.getbbox()  # Get bounding box of non-zero region in mask
                if bbox:
                    object_img_cropped = object_img.crop(bbox)

                    # Convert RGBA to RGB before saving as JPEG
                    object_img_cropped_rgb = object_img_cropped.convert("RGB")

                    # Save cropped image without transparency
                    object_img_cropped_rgb.save(os.path.join(image_dest_path, image_filename))
        else:
            print(f"Image file not found: {image_path}")




In [ ]:
#Usage
def main():
    #Parameters
    json_source_path = 'original_json'#Replace with you own path for images
    image_source_path = 'original_img'#Replace with you own path for images
    image_dest_path = './img_cropped'#Replace with you own path for images
    category_id_to_crop = 55

    try:
        # Process JSON files to get cropping information (segmentation points)
        crop_info = process_json_files(json_source_path, category_id_to_crop)

        if not crop_info:
            print(f"No annotations found with category_id {category_id_to_crop}")
            return

        # Crop images based on segmentation and save them without transparency
        crop_images(crop_info, image_source_path, image_dest_path)

        print("Cropping completed successfully.")

    except FileNotFoundError as e:
        print(e)